# 12.08 - CLIP image-text retrieval

**Notebook type:** Solution notebook with completed exercises, smoke checks, and test cases.

**Daily output:** Bidirectional retrieval demo and Recall@K report.

Build both image-to-text and text-to-image ranking from cached-like normalized embeddings, then inspect failures instead of reporting only aggregate Recall@K.

## Core Ideas

Retrieval ranks every candidate by cosine similarity. Image-to-text and text-to-image are different queries and should be evaluated separately. Recall@K asks whether the matching ID appears in the first K results. Cache embeddings once, maintain stable IDs, and normalize before matrix multiplication.

In [ ]:
import numpy as np
import pandas as pd
import torch

SEED = 12
np.random.seed(SEED)
torch.manual_seed(SEED)

## Prepared Image–Text Gallery

Eight matched IDs have paired four-dimensional embeddings. Two pairs are intentionally noisy enough to produce useful ranking errors.

In [ ]:
gallery_ids = [f"item_{index:02d}" for index in range(8)]
base_embeddings = torch.randn(8, 4)
image_embeddings = base_embeddings + 0.05 * torch.randn(8, 4)
text_embeddings = base_embeddings + 0.05 * torch.randn(8, 4)
text_embeddings[6] = text_embeddings[5] + 0.02
text_embeddings[7] = text_embeddings[6] + 0.02
print("gallery:", len(gallery_ids), image_embeddings.shape, text_embeddings.shape)

## Exercise 12-A: Build a similarity matrix

Normalize both modalities and calculate all pairwise cosine similarities.

**Return structure — `cross_modal_similarity`:** A CPU float32 tensor `[N_image,N_text]`; rows are image queries and columns are text candidates.

In [ ]:
def cross_modal_similarity(images, texts):
    images = torch.nn.functional.normalize(images.detach().cpu().float(), dim=1)
    texts = torch.nn.functional.normalize(texts.detach().cpu().float(), dim=1)
    if images.shape[1] != texts.shape[1]:
        raise ValueError("embedding dimensions must match")
    return images @ texts.T


# Smoke check: score every gallery pair.
similarity_matrix = cross_modal_similarity(image_embeddings, text_embeddings)
print(similarity_matrix.shape, similarity_matrix.diag())

## Exercise 12-B: Rank top-k candidates

Return stable candidate IDs and scores for every query.

**Return structure — `rank_candidates`:** A `list[dict]` with one row per query. Keys are `query_id` (`str`), `candidate_ids` (`list[str]` length `k`), and `scores` (`list[float]` length `k`).

In [ ]:
def rank_candidates(similarities, query_ids, candidate_ids, k=3):
    values, indices = similarities.topk(min(int(k), similarities.shape[1]), dim=1)
    return [{"query_id": str(query_ids[row]), "candidate_ids": [str(candidate_ids[index]) for index in indices[row].tolist()], "scores": [float(value) for value in values[row].tolist()]} for row in range(len(query_ids))]


# Smoke check: rank both retrieval directions.
image_to_text_rows = rank_candidates(similarity_matrix, gallery_ids, gallery_ids, k=3)
text_to_image_rows = rank_candidates(similarity_matrix.T, gallery_ids, gallery_ids, k=3)
print(image_to_text_rows[:2])

## Exercise 12-C: Calculate Recall@K

A query succeeds when its matching stable ID appears among the first K candidates.

**Return structure — `recall_at_k`:** A Python float in `[0,1]`, calculated over every retrieval row.

In [ ]:
def recall_at_k(ranking_rows, k):
    if not ranking_rows:
        raise ValueError("ranking_rows must not be empty")
    return float(np.mean([row["query_id"] in row["candidate_ids"][:int(k)] for row in ranking_rows]))


# Smoke check: calculate both directions at K=1 and K=3.
retrieval_metrics = {"i2t_r1": recall_at_k(image_to_text_rows, 1), "i2t_r3": recall_at_k(image_to_text_rows, 3), "t2i_r1": recall_at_k(text_to_image_rows, 1), "t2i_r3": recall_at_k(text_to_image_rows, 3)}
print(retrieval_metrics)

## Exercise 12-D: Create a failure table

Retain the expected ID, top candidate, and top score for failed top-1 queries.

**Return structure — `retrieval_failures`:** A DataFrame with columns `direction`, `query_id`, `expected_id`, `top_candidate`, and `top_score`; it may have zero rows.

In [ ]:
def retrieval_failures(named_rankings):
    rows = []
    for direction, rankings in named_rankings.items():
        for row in rankings:
            if row["candidate_ids"][0] != row["query_id"]:
                rows.append({"direction": direction, "query_id": row["query_id"], "expected_id": row["query_id"], "top_candidate": row["candidate_ids"][0], "top_score": row["scores"][0]})
    return pd.DataFrame(rows, columns=["direction", "query_id", "expected_id", "top_candidate", "top_score"])


# Smoke check: inspect failures in both directions.
retrieval_error_table = retrieval_failures({"image_to_text": image_to_text_rows, "text_to_image": text_to_image_rows})
print(retrieval_error_table.to_string(index=False))

## Test Cases

**Return structure — `run_day12_tests`:** Returns `None`; assertions and `Day 12 tests passed` communicate success.

In [ ]:
def run_day12_tests():
    assert similarity_matrix.shape == (8, 8) and similarity_matrix.dtype == torch.float32
    assert len(image_to_text_rows) == len(text_to_image_rows) == 8
    assert all(len(row["candidate_ids"]) == len(row["scores"]) == 3 for row in image_to_text_rows)
    assert all(0.0 <= value <= 1.0 for value in retrieval_metrics.values())
    assert retrieval_metrics["i2t_r3"] >= retrieval_metrics["i2t_r1"]
    assert retrieval_metrics["t2i_r3"] >= retrieval_metrics["t2i_r1"]
    assert list(retrieval_error_table.columns) == ["direction", "query_id", "expected_id", "top_candidate", "top_score"]
    print("Day 12 tests passed")


run_day12_tests()

## Day 12 Checklist

- [ ] Normalize both embedding modalities.
- [ ] Preserve stable gallery identifiers.
- [ ] Evaluate both retrieval directions.
- [ ] Inspect top-1 failures beside Recall@K.
- [ ] Run the test cases.